<a href="https://colab.research.google.com/github/fida22/nighttime-flare-removal/blob/main/DS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Can you all see my comments???

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

PROJECT = "/content/drive/MyDrive/Flare_Removal_Project"

print("PROJECT FOLDER:")
for item in os.listdir(PROJECT):
    print(item)

print("\nARCHIVES FOLDER:")
archive_dir = os.path.join(PROJECT, "data", "archives")

for item in os.listdir(archive_dir):
    print(item)

PROJECT FOLDER:
data
checkpoints
logs
results
Flare7K_Team_Complete_Code_and_Colab_Notebooks.zip

ARCHIVES FOLDER:
Flickr24K.zip
Flare7K++.zip


In [ ]:
import os

for root, dirs, files in os.walk(PROJECT):
    for file in files:
        if file.lower().endswith(".zip"):
            print(os.path.join(root, file))

/content/drive/MyDrive/Flare_Removal_Project/Flare7K_Team_Complete_Code_and_Colab_Notebooks.zip
/content/drive/MyDrive/Flare_Removal_Project/data/archives/Flickr24K.zip
/content/drive/MyDrive/Flare_Removal_Project/data/archives/Flare7K++.zip


In [ ]:
import os
import zipfile

PROJECT = "/content/drive/MyDrive/Flare_Removal_Project"

CODE_ZIP = os.path.join(
    PROJECT,
    "Flare7K_Team_Complete_Code_and_Colab_Notebooks.zip"
)

CODE_DIR = "/content"

with zipfile.ZipFile(CODE_ZIP, "r") as z:
    z.extractall(CODE_DIR)

print("Project code extracted!")
print("Folder exists:", os.path.exists("/content/nighttime-flare-removal"))

Project code extracted!
Folder exists: True


In [ ]:
import os
import zipfile

PROJECT = "/content/drive/MyDrive/Flare_Removal_Project"

FLARE_ZIP = os.path.join(
    PROJECT, "data", "archives", "Flare7K++.zip"
)

FLICKR_ZIP = os.path.join(
    PROJECT, "data", "archives", "Flickr24K.zip"
)

EXTRACT_DIR = "/content/flare_data"

os.makedirs(EXTRACT_DIR, exist_ok=True)

# Extract Flare7K++
flare_destination = os.path.join(EXTRACT_DIR, "Flare7K++")

if not os.path.exists(flare_destination):
    with zipfile.ZipFile(FLARE_ZIP, "r") as z:
        z.extractall(flare_destination)

print("Flare7K++ restored!")

# Extract Flickr24K
flickr_destination = os.path.join(EXTRACT_DIR, "Flickr24K")

if not os.path.exists(flickr_destination):
    with zipfile.ZipFile(FLICKR_ZIP, "r") as z:
        z.extractall(flickr_destination)

print("Flickr24K restored!")

Flare7K++ restored!
Flickr24K restored!


In [ ]:
import os
import sys

OFFICIAL_REPO = "/content/Flare7K"

if not os.path.exists(OFFICIAL_REPO):
    !git clone https://github.com/ykdai/Flare7K.git /content/Flare7K

if OFFICIAL_REPO not in sys.path:
    sys.path.insert(0, OFFICIAL_REPO)

from data_loader import Flare_Image_Loader

print("Official Flare7K loader restored!")

Cloning into '/content/Flare7K'...
remote: Enumerating objects: 349, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 349 (delta 50), reused 32 (delta 14), pack-reused 251 (from 1)
Receiving objects: 100% (349/349), 4.78 MiB | 21.47 MiB/s, done.
Resolving deltas: 100% (126/126), done.
Official Flare7K loader restored!


In [ ]:
import os

FLARE_ROOT = "/content/flare_data/Flare7K++/Flare7Kpp"
FLICKR_ROOT = "/content/flare_data/Flickr24K/Flickr24K"

REPO = "/content/nighttime-flare-removal"

print("========== RESTORATION STATUS ==========")

print("Project code:", os.path.exists(REPO))
print("Official loader:", os.path.exists("/content/Flare7K/data_loader.py"))
print("Flare7K++:", os.path.exists(FLARE_ROOT))
print("Flickr24K:", os.path.exists(FLICKR_ROOT))

if os.path.exists(FLICKR_ROOT):
    print("Flickr images:", len(os.listdir(FLICKR_ROOT)))

========== RESTORATION STATUS ==========
Project code: True
Official loader: True
Flare7K++: True
Flickr24K: True
Flickr images: 23949


In [ ]:
import os
import sys
import torch
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

OFFICIAL_REPO = "/content/Flare7K"
PROJECT_CODE = "/content/nighttime-flare-removal"

sys.path.insert(0, OFFICIAL_REPO)
sys.path.insert(0, PROJECT_CODE)

from data_loader import Flare_Image_Loader

print("All imports successful!")

All imports successful!


In [ ]:
FLARE_ROOT = "/content/flare_data/Flare7K++/Flare7Kpp"
FLICKR_ROOT = "/content/flare_data/Flickr24K/Flickr24K"

PROJECT = "/content/drive/MyDrive/Flare_Removal_Project"

CHECKPOINT_DIR = os.path.join(PROJECT, "checkpoints")
RESULT_DIR = os.path.join(PROJECT, "results")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

print("Dataset paths configured!")

Dataset paths configured!


In [ ]:
transform_base = transforms.Compose([
    transforms.RandomCrop(
        (512, 512),
        pad_if_needed=True,
        padding_mode="reflect"
    ),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip()
])

transform_flare = transforms.Compose([
    transforms.RandomAffine(
        degrees=(0, 360),
        scale=(0.8, 1.5),
        translate=(300/1440, 300/1440),
        shear=(-20, 20)
    ),
    transforms.CenterCrop((512, 512)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip()
])

print("Transformations ready!")

Transformations ready!


In [ ]:
flare_image_loader = Flare_Image_Loader(
    FLICKR_ROOT,
    transform_base,
    transform_flare,
    mask_type="luminance"
)

flare_image_loader.load_scattering_flare(
    "Flare7K",
    os.path.join(
        FLARE_ROOT,
        "Flare7K",
        "Scattering_Flare",
        "Compound_Flare"
    )
)

flare_image_loader.load_reflective_flare(
    "Flare7K",
    os.path.join(
        FLARE_ROOT,
        "Flare7K",
        "Reflective_Flare"
    )
)

Base Image Loaded with examples: 23949
Scattering Flare Image: Flare7K  is loaded successfully with examples 5000
Now we have 5000 scattering flare images
Reflective Flare Image: Flare7K  is loaded successfully with examples 2000
Now we have 2000 refelctive flare images


In [ ]:
flare_image_loader = Flare_Image_Loader(
    FLICKR_ROOT,
    transform_base,
    transform_flare,
    mask_type="luminance"
)

flare_image_loader.load_scattering_flare(
    "Flare7K",
    os.path.join(
        FLARE_ROOT,
        "Flare7K",
        "Scattering_Flare",
        "Compound_Flare"
    )
)

flare_image_loader.load_reflective_flare(
    "Flare7K",
    os.path.join(
        FLARE_ROOT,
        "Flare7K",
        "Reflective_Flare"
    )
)

Base Image Loaded with examples: 23949
Scattering Flare Image: Flare7K  is loaded successfully with examples 5000
Now we have 5000 scattering flare images
Reflective Flare Image: Flare7K  is loaded successfully with examples 2000
Now we have 2000 refelctive flare images


In [ ]:
from torch.utils.data import DataLoader, random_split

total_size = len(flare_image_loader)

train_size = int(0.8 * total_size)
val_size = total_size - train_size

generator = torch.Generator().manual_seed(42)

train_dataset, val_dataset = random_split(
    flare_image_loader,
    [train_size, val_size],
    generator=generator
)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Training samples: 19159
Validation samples: 4790
Training batches: 4790
Validation batches: 1198


In [ ]:
batch = next(iter(train_loader))

clean_images, flare_images, input_images, masks, gammas = batch

print("Clean images:", clean_images.shape)
print("Flare images:", flare_images.shape)
print("Input images:", input_images.shape)
print("Masks:", masks.shape)

Clean images: torch.Size([4, 3, 512, 512])
Flare images: torch.Size([4, 3, 512, 512])
Input images: torch.Size([4, 3, 512, 512])
Masks: torch.Size([4, 3, 512, 512])


In [ ]:
import torch
import torch.nn as nn

# Two convolution layers
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


# Baseline U-Net
class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=3):
        super().__init__()

        # Encoder
        self.enc1 = DoubleConv(in_channels, 32)
        self.enc2 = DoubleConv(32, 64)
        self.enc3 = DoubleConv(64, 128)
        self.enc4 = DoubleConv(128, 256)

        self.pool = nn.MaxPool2d(kernel_size=2)

        # Bottleneck
        self.bottleneck = DoubleConv(256, 512)

        # Decoder
        self.up4 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(512, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(64, 32)

        # Final output layer
        self.final = nn.Conv2d(32, out_channels, kernel_size=1)

    def forward(self, x):

        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        # Bottleneck
        b = self.bottleneck(self.pool(e4))

        # Decoder + skip connections
        d4 = self.up4(b)
        d4 = self.dec4(torch.cat([d4, e4], dim=1))

        d3 = self.up3(d4)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))

        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return torch.sigmoid(self.final(d1))


print("Baseline U-Net architecture created!")

Baseline U-Net architecture created!


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet(
    in_channels=3,
    out_channels=3
).to(device)

print("Device:", device)
print("Model created successfully!")

total_params = sum(p.numel() for p in model.parameters())

print("Total parameters:", f"{total_params:,}")

Device: cuda
Model created successfully!
Total parameters: 7,766,051


In [ ]:
test_input = input_images.to(device)

with torch.no_grad():
    prediction = model(test_input)

print("Input shape:", test_input.shape)
print("Output shape:", prediction.shape)

Input shape: torch.Size([4, 3, 512, 512])
Output shape: torch.Size([4, 3, 512, 512])


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Loss function
criterion = nn.L1Loss()

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# Number of epochs
EPOCHS = 20

print("Loss function:", criterion)
print("Optimizer:", optimizer)
print("Epochs:", EPOCHS)

Loss function: L1Loss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0001
    maximize: False
    weight_decay: 0
)
Epochs: 20


In [ ]:
print("========== RECOVERY CHECK ==========")

print("Model available:", "model" in globals())
print("Optimizer available:", "optimizer" in globals())
print("Device available:", "device" in globals())
print("Epoch variable available:", "epoch" in globals())

if "epoch" in globals():
    print("Last epoch variable:", epoch + 1)

print("====================================")

========== RECOVERY CHECK ==========
Model available: True
Optimizer available: True
Device available: True
Epoch variable available: False


In [ ]:
print("========== CURRENT STATUS ==========")

print("Model:", "model" in globals())
print("Optimizer:", "optimizer" in globals())
print("Device:", "device" in globals())
print("Epoch:", "epoch" in globals())
print("Project:", "PROJECT" in globals())

print("====================================")

========== CURRENT STATUS ==========
Model: True
Optimizer: True
Device: True
Epoch: False
Project: True


In [ ]:
import os
import torch

checkpoint_path = os.path.join(
    PROJECT,
    "checkpoints",
    "baseline",
    "baseline_unet_recovered.pth"
)

# Check that the checkpoint exists
if os.path.exists(checkpoint_path):

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=False
    )

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    # Restore epoch information
    start_epoch = checkpoint["epoch"]

    print("Checkpoint loaded successfully!")
    print("Saved epoch index:", start_epoch)
    print("Next training start epoch:", start_epoch + 1)

else:
    print("Checkpoint not found! Do not start training yet.")

Checkpoint loaded successfully!
Saved epoch index: 0
Next training start epoch: 1


In [ ]:
import os
import torch

checkpoint_dir = os.path.join(
    PROJECT, "checkpoints", "baseline"
)

os.makedirs(checkpoint_dir, exist_ok=True)

def save_checkpoint(epoch, batch_number, loss):

    checkpoint = {
        "epoch": epoch,
        "batch_number": batch_number,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "loss": loss,
        "total_epochs": EPOCHS
    }

    save_path = os.path.join(
        checkpoint_dir, "baseline_unet_latest.pth"
    )

    temp_path = save_path + ".tmp"

    torch.save(checkpoint, temp_path)
    os.replace(temp_path, save_path)

    print(f"Checkpoint saved: Epoch {epoch}, Batch {batch_number}")

In [ ]:
EPOCHS = 20
SAVE_EVERY = 100

for epoch in range(start_epoch, EPOCHS):

    model.train()
    train_loss = 0.0

    for batch_number, batch in enumerate(train_loader, start=1):

        base_img, flare_img, merge_img, flare_mask, gamma = batch

        inputs = merge_img.to(device)
        targets = base_img.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        # Save every 100 batches
        if batch_number % SAVE_EVERY == 0:
            save_checkpoint(
                epoch, batch_number, loss.item()
            )

    average_loss = train_loss / len(train_loader)

    print(
        f"Epoch [{epoch + 1}/{EPOCHS}] "
        f"Training Loss: {average_loss:.4f}"
    )

    # Save after each completed epoch
    save_checkpoint(epoch + 1, 0, average_loss)

print("Baseline U-Net training completed!")

Checkpoint saved: Epoch 0, Batch 100
Checkpoint saved: Epoch 0, Batch 200
Checkpoint saved: Epoch 0, Batch 300
Checkpoint saved: Epoch 0, Batch 400
Checkpoint saved: Epoch 0, Batch 500
Checkpoint saved: Epoch 0, Batch 600
Checkpoint saved: Epoch 0, Batch 700
Checkpoint saved: Epoch 0, Batch 800
Checkpoint saved: Epoch 0, Batch 900
Checkpoint saved: Epoch 0, Batch 1000
Checkpoint saved: Epoch 0, Batch 1100
Checkpoint saved: Epoch 0, Batch 1200
Checkpoint saved: Epoch 0, Batch 1300
Checkpoint saved: Epoch 0, Batch 1400
Checkpoint saved: Epoch 0, Batch 1500
Checkpoint saved: Epoch 0, Batch 1600
Checkpoint saved: Epoch 0, Batch 1700
Checkpoint saved: Epoch 0, Batch 1800
Checkpoint saved: Epoch 0, Batch 1900
Checkpoint saved: Epoch 0, Batch 2000
Checkpoint saved: Epoch 0, Batch 2100


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import sys
import torch

PROJECT = "/content/drive/MyDrive/Flare_Removal_Project"

FLARE_ROOT = "/content/flare_data/Flare7K++/Flare7Kpp"
FLICKR_ROOT = "/content/flare_data/Flickr24K/Flickr24K"

OFFICIAL_REPO = "/content/Flare7K"
PROJECT_CODE = "/content/nighttime-flare-removal"

CHECKPOINT_DIR = os.path.join(PROJECT, "checkpoints")
RESULT_DIR = os.path.join(PROJECT, "results")

print("Google Drive:", os.path.exists(PROJECT))
print("Flare7K++:", os.path.exists(FLARE_ROOT))
print("Flickr24K:", os.path.exists(FLICKR_ROOT))
print("Official loader:", os.path.exists(
    os.path.join(OFFICIAL_REPO, "data_loader.py")
))
print("GPU:", torch.cuda.is_available())

Google Drive: True
Flare7K++: False
Flickr24K: False
Official loader: False
GPU: True


In [ ]:
import os

PROJECT = "/content/drive/MyDrive/Flare_Removal_Project"

for root, dirs, files in os.walk(PROJECT):
    for file in files:
        if file.endswith(".zip"):
            print(os.path.join(root, file))

/content/drive/MyDrive/Flare_Removal_Project/Flare7K_Team_Complete_Code_and_Colab_Notebooks.zip
/content/drive/MyDrive/Flare_Removal_Project/data/archives/Flickr24K.zip
/content/drive/MyDrive/Flare_Removal_Project/data/archives/Flare7K++.zip


In [ ]:
import os
import zipfile

DATA_DIR = "/content/flare_data"
os.makedirs(DATA_DIR, exist_ok=True)

archives = {
    "Flare7K++": os.path.join(
        PROJECT, "data", "archives", "Flare7K++.zip"
    ),
    "Flickr24K": os.path.join(
        PROJECT, "data", "archives", "Flickr24K.zip"
    )
}

for name, zip_path in archives.items():

    if os.path.exists(zip_path):
        print("Extracting:", name)

        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(DATA_DIR)

        print(name, "extracted successfully!")

    else:
        print("ZIP NOT FOUND:", zip_path)

Extracting: Flare7K++
Flare7K++ extracted successfully!
Extracting: Flickr24K
Flickr24K extracted successfully!


In [ ]:
!git clone https://github.com/ykdai/Flare7K.git /content/Flare7K

Cloning into '/content/Flare7K'...
remote: Enumerating objects: 349, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 349 (delta 50), reused 32 (delta 14), pack-reused 251 (from 1)
Receiving objects: 100% (349/349), 4.78 MiB | 10.39 MiB/s, done.
Resolving deltas: 100% (126/126), done.


In [ ]:
import os
import torch

FLARE_ROOT = "/content/flare_data/Flare7K++/Flare7Kpp"
FLICKR_ROOT = "/content/flare_data/Flickr24K/Flickr24K"
OFFICIAL_REPO = "/content/Flare7K"

print("========== RESTORATION CHECK ==========")
print("Flare7K++:", os.path.exists(FLARE_ROOT))
print("Flickr24K:", os.path.exists(FLICKR_ROOT))
print("Official loader:", os.path.exists(
    os.path.join(OFFICIAL_REPO, "data_loader.py")
))
print("GPU available:", torch.cuda.is_available())

print("=======================================")

========== RESTORATION CHECK ==========
Flare7K++: False
Flickr24K: False
Official loader: True
GPU available: True


In [ ]:
import os

DATA_DIR = "/content/flare_data"

print("Folders inside flare_data:\n")

for item in os.listdir(DATA_DIR):
    print(" -", item)

print("\nSearching for dataset folders...\n")

for root, dirs, files in os.walk(DATA_DIR):
    for folder in dirs:
        if folder in ["Flare7Kpp", "Flickr24K"]:
            print("Found:", os.path.join(root, folder))

Folders inside flare_data:

 - Flickr24K
 - Flare7Kpp

Searching for dataset folders...

Found: /content/flare_data/Flickr24K
Found: /content/flare_data/Flare7Kpp


In [ ]:
FLARE_ROOT = "/content/flare_data/Flare7Kpp"
FLICKR_ROOT = "/content/flare_data/Flickr24K"

import os

print("Flare7K++:", os.path.exists(FLARE_ROOT))
print("Flickr24K:", os.path.exists(FLICKR_ROOT))
print("Official loader:", os.path.exists("/content/Flare7K/data_loader.py"))

Flare7K++: True
Flickr24K: True
Official loader: True


In [ ]:
import os
import torch

PROJECT = "/content/drive/MyDrive/Flare_Removal_Project"

checkpoint_path = os.path.join(
    PROJECT,
    "checkpoints",
    "baseline",
    "baseline_unet_latest.pth"
)

print("Checkpoint exists:", os.path.exists(checkpoint_path))

if os.path.exists(checkpoint_path):
    ckpt = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False
    )

    print("Saved epoch:", ckpt["epoch"])
    print("Saved batch:", ckpt["batch_number"])
    print("Saved loss:", ckpt["loss"])

Checkpoint exists: True
Saved epoch: 0
Saved batch: 2100
Saved loss: 0.05862467736005783


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=3):
        super().__init__()

        self.enc1 = DoubleConv(in_channels, 32)
        self.enc2 = DoubleConv(32, 64)
        self.enc3 = DoubleConv(64, 128)
        self.enc4 = DoubleConv(128, 256)

        self.pool = nn.MaxPool2d(kernel_size=2)

        self.bottleneck = DoubleConv(256, 512)

        self.up4 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.dec4 = DoubleConv(512, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec3 = DoubleConv(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec2 = DoubleConv(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, 2, 2)
        self.dec1 = DoubleConv(64, 32)

        self.final = nn.Conv2d(32, out_channels, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.bottleneck(self.pool(e4))

        d4 = self.up4(b)
        d4 = self.dec4(torch.cat([d4, e4], dim=1))

        d3 = self.up3(d4)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))

        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return torch.sigmoid(self.final(d1))


print("U-Net architecture recreated successfully!")

U-Net architecture recreated successfully!


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet(in_channels=3, out_channels=3).to(device)

criterion = nn.L1Loss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

EPOCHS = 20

print("Model initialized!")
print("Device:", device)

Model initialized!
Device: cuda


In [ ]:
checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

start_epoch = checkpoint["epoch"]

print("Model restored successfully!")
print("Saved epoch:", start_epoch)
print("Saved batch:", checkpoint["batch_number"])
print("Saved loss:", checkpoint["loss"])

Model restored successfully!
Saved epoch: 0
Saved batch: 2100
Saved loss: 0.05862467736005783


In [ ]:
print("========== TRAINING RESTORATION ==========")

print("Model:", "Ready" if "model" in globals() else "Missing")
print("Optimizer:", "Ready" if "optimizer" in globals() else "Missing")
print("Train loader:", "Ready" if "train_loader" in globals() else "Missing")
print("Validation loader:", "Ready" if "val_loader" in globals() else "Missing")
print("Checkpoint directory:", "Ready" if "checkpoint_dir" in globals() else "Missing")

print("==========================================")

========== TRAINING RESTORATION ==========
Model: Ready
Optimizer: Ready
Train loader: Missing
Validation loader: Missing
Checkpoint directory: Missing


In [ ]:
import os
import sys
import torch
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split

# Dataset locations
FLARE_ROOT = "/content/flare_data/Flare7Kpp"
FLICKR_ROOT = "/content/flare_data/Flickr24K"
OFFICIAL_REPO = "/content/Flare7K"

# Check folder contents
print("Flare7K++ contents:")
print(os.listdir(FLARE_ROOT)[:15])

print("\nFlickr24K contents:")
print(os.listdir(FLICKR_ROOT)[:10])

# Check official loader
loader_file = os.path.join(OFFICIAL_REPO, "data_loader.py")
print("\nOfficial loader exists:", os.path.exists(loader_file))

Flare7K++ contents:
['Flare-R', 'Flare7K', 'test_data']

Flickr24K contents:
['r (4607).jpg', 'r (465).jpg', 'r (8356).jpg', '14751.jpg', '6638.jpg', 'r (440).jpg', '7449.jpg', '4462.jpg', 'r (4597).jpg', 'r (10592).jpg']

Official loader exists: True


In [ ]:
import sys
import os

sys.path.insert(0, "/content/Flare7K")

from data_loader import Flare_Image_Loader

print("Official Flare7K loader imported successfully!")

Official Flare7K loader imported successfully!


In [ ]:
import torchvision.transforms as transforms

transform_base = transforms.Compose([
    transforms.RandomCrop(
        (512, 512),
        pad_if_needed=True,
        padding_mode="reflect"
    ),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip()
])

transform_flare = transforms.Compose([
    transforms.RandomAffine(
        degrees=(0, 360),
        scale=(0.8, 1.5),
        translate=(300/1440, 300/1440),
        shear=(-20, 20)
    ),
    transforms.CenterCrop((512, 512)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip()
])

print("Image transformations created successfully!")

Image transformations created successfully!


In [ ]:
import os

flare_image_loader = Flare_Image_Loader(
    FLICKR_ROOT,
    transform_base,
    transform_flare,
    mask_type="luminance"
)

# Load scattering flare patterns
flare_image_loader.load_scattering_flare(
    "Flare7K",
    os.path.join(
        FLARE_ROOT,
        "Flare7K",
        "Scattering_Flare",
        "Compound_Flare"
    )
)

# Load reflective flare patterns
flare_image_loader.load_reflective_flare(
    "Flare7K",
    os.path.join(
        FLARE_ROOT,
        "Flare7K",
        "Reflective_Flare"
    )
)

print("Flare7K++ dataset recreated successfully!")
print("Total available samples:", len(flare_image_loader))

Base Image Loaded with examples: 23949
Scattering Flare Image: Flare7K  is loaded successfully with examples 5000
Now we have 5000 scattering flare images
Reflective Flare Image: Flare7K  is loaded successfully with examples 2000
Now we have 2000 refelctive flare images
Flare7K++ dataset recreated successfully!
Total available samples: 23949


In [ ]:
from torch.utils.data import DataLoader, random_split

# 80% training, 20% validation
total_size = len(flare_image_loader)

train_size = int(0.8 * total_size)
val_size = total_size - train_size

generator = torch.Generator().manual_seed(42)

train_dataset, val_dataset = random_split(
    flare_image_loader,
    [train_size, val_size],
    generator=generator
)

# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Training and validation loaders recreated!")
print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Training batches per epoch:", len(train_loader))
print("Validation batches:", len(val_loader))

Training and validation loaders recreated!
Training samples: 19159
Validation samples: 4790
Training batches per epoch: 4790
Validation batches: 1198


In [ ]:
import os

PROJECT = "/content/drive/MyDrive/Flare_Removal_Project"

checkpoint_dir = os.path.join(
    PROJECT,
    "checkpoints",
    "baseline"
)

os.makedirs(checkpoint_dir, exist_ok=True)

checkpoint_path = os.path.join(
    checkpoint_dir,
    "baseline_unet_latest.pth"
)

print("Checkpoint directory ready:", checkpoint_dir)
print("Existing checkpoint preserved:", os.path.exists(checkpoint_path))

Checkpoint directory ready: /content/drive/MyDrive/Flare_Removal_Project/checkpoints/baseline
Existing checkpoint preserved: True


In [ ]:
print("========== READY TO TRAIN ==========")

print("GPU:", torch.cuda.get_device_name(0))
print("Model:", type(model).__name__)
print("Optimizer:", type(optimizer).__name__)
print("Training batches:", len(train_loader))
print("Checkpoint exists:", os.path.exists(checkpoint_path))

print("===================================")

========== READY TO TRAIN ==========
GPU: Tesla T4
Model: UNet
Optimizer: Adam
Training batches: 4790
Checkpoint exists: True


In [ ]:
import os
import shutil
import torch
from datetime import datetime

checkpoint_dir = os.path.join(
    PROJECT, "checkpoints", "baseline"
)

os.makedirs(checkpoint_dir, exist_ok=True)

checkpoint_path = os.path.join(
    checkpoint_dir, "baseline_unet_latest.pth"
)

backup_path = os.path.join(
    checkpoint_dir, "baseline_unet_backup.pth"
)

# Preserve the currently recovered checkpoint
if not os.path.exists(backup_path):
    shutil.copy2(checkpoint_path, backup_path)

print("Automatic checkpoint system initialized!")
print("Latest:", checkpoint_path)
print("Backup:", backup_path)
print("Existing training progress preserved!")

Automatic checkpoint system initialized!
Latest: /content/drive/MyDrive/Flare_Removal_Project/checkpoints/baseline/baseline_unet_latest.pth
Backup: /content/drive/MyDrive/Flare_Removal_Project/checkpoints/baseline/baseline_unet_backup.pth
Existing training progress preserved!


In [ ]:
import os
import shutil

SAVE_EVERY = 100
BACKUP_EVERY = 500

def save_checkpoint(epoch, batch_number, loss):

    checkpoint = {
        "epoch": epoch,
        "batch_number": batch_number,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "loss": loss,
        "total_epochs": EPOCHS,
        "saved_at": datetime.now().isoformat()
    }

    temp_path = checkpoint_path + ".tmp"

    # Save latest checkpoint safely
    torch.save(checkpoint, temp_path)
    os.replace(temp_path, checkpoint_path)

    print(
        f"Checkpoint saved | "
        f"Epoch: {epoch + 1} | "
        f"Batch: {batch_number} | "
        f"Loss: {loss:.6f}"
    )

    # Create an independent backup periodically
    if batch_number == 0 or batch_number % BACKUP_EVERY == 0:

        backup_temp = backup_path + ".tmp"

        try:
            shutil.copy2(checkpoint_path, backup_temp)
            os.replace(backup_temp, backup_path)
            print("Backup checkpoint updated!")

        except Exception as e:
            print("Backup update warning:", e)

print("Checkpoint function ready!")

Checkpoint function ready!


In [ ]:
# Read the latest saved progress
checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

start_epoch = checkpoint["epoch"]
resume_batch = checkpoint["batch_number"]

print("========== RESUMING TRAINING ==========")
print("Starting epoch:", start_epoch + 1)
print("Previously saved batch:", resume_batch)
print("=======================================")

last_completed_batch = resume_batch
last_loss = checkpoint["loss"]

try:

    for epoch in range(start_epoch, EPOCHS):

        model.train()

        train_loss = 0.0
        processed_batches = 0

        print(f"\nStarting Epoch {epoch + 1}/{EPOCHS}")

        for batch_number, batch in enumerate(train_loader, start=1):

            # Skip previously processed batch positions
            if epoch == start_epoch and batch_number <= resume_batch:
                continue

            base_img, flare_img, merge_img, flare_mask, gamma = batch

            inputs = merge_img.to(device)
            targets = base_img.to(device)

            optimizer.zero_grad()

            outputs = model(inputs)

            loss = criterion(outputs, targets)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            processed_batches += 1

            last_completed_batch = batch_number
            last_loss = loss.item()

            # Save every 100 batches
            if batch_number % SAVE_EVERY == 0:

                save_checkpoint(
                    epoch,
                    batch_number,
                    loss.item()
                )

        # Calculate loss for batches processed in this run
        if processed_batches > 0:
            average_loss = train_loss / processed_batches
        else:
            average_loss = 0.0

        print(
            f"\nEpoch {epoch + 1} completed!"
            f"\nRemaining-batch average loss: {average_loss:.6f}"
        )

        # Save completed epoch
        save_checkpoint(
            epoch + 1,
            0,
            average_loss
        )

        # Next epoch starts from batch zero
        resume_batch = 0

    print("\nALL TRAINING EPOCHS COMPLETED!")

except KeyboardInterrupt:

    print("\nTraining manually interrupted.")

    save_checkpoint(
        epoch,
        last_completed_batch,
        last_loss
    )

    print("Latest progress saved!")

except Exception as e:

    print("\nTraining encountered an error:", e)

    save_checkpoint(
        epoch,
        last_completed_batch,
        last_loss
    )

    print("Attempted to save the latest progress.")
    raise

========== RESUMING TRAINING ==========
Starting epoch: 1
Previously saved batch: 2100

Starting Epoch 1/20
Checkpoint saved | Epoch: 1 | Batch: 2200 | Loss: 0.066341
Checkpoint saved | Epoch: 1 | Batch: 2300 | Loss: 0.060854
Checkpoint saved | Epoch: 1 | Batch: 2400 | Loss: 0.078316
Checkpoint saved | Epoch: 1 | Batch: 2500 | Loss: 0.053835
Backup checkpoint updated!


In [ ]:
import os

PROJECT = "/content/drive/MyDrive/Flare_Removal_Project"

checkpoint_path = os.path.join(
    PROJECT,
    "checkpoints",
    "baseline",
    "baseline_unet_latest.pth"
)

print("Checkpoint exists:", os.path.exists(checkpoint_path))

Checkpoint exists: False


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

drive_root = "/content/drive/MyDrive"

print("Searching Google Drive for baseline checkpoints...\n")

found = []

for root, dirs, files in os.walk(drive_root):
    dirs[:] = [d for d in dirs if d not in [
        ".Trash", ".ipynb_checkpoints"
    ]]

    for filename in files:
        if filename in [
            "baseline_unet_latest.pth",
            "baseline_unet_recovered.pth",
            "baseline_unet_backup.pth"
        ]:
            path = os.path.join(root, filename)
            size_mb = os.path.getsize(path) / (1024 * 1024)
            found.append(path)
            print(f"FOUND: {path}")
            print(f"SIZE: {size_mb:.2f} MB\n")

if not found:
    print("No matching checkpoint files found.")

Searching Google Drive for baseline checkpoints...

No matching checkpoint files found.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

drive_root = "/content/drive/MyDrive"

print("Searching Google Drive for baseline checkpoints...\n")

found = []

for root, dirs, files in os.walk(drive_root):
    dirs[:] = [d for d in dirs if d not in [
        ".Trash", ".ipynb_checkpoints"
    ]]

    for filename in files:
        if filename in [
            "baseline_unet_latest.pth",
            "baseline_unet_recovered.pth",
            "baseline_unet_backup.pth"
        ]:
            path = os.path.join(root, filename)
            size_mb = os.path.getsize(path) / (1024 * 1024)
            found.append(path)
            print(f"FOUND: {path}")
            print(f"SIZE: {size_mb:.2f} MB\n")

if not found:
    print("No matching checkpoint files found.")

Searching Google Drive for baseline checkpoints...

No matching checkpoint files found.


In [ ]:
import os

print("Drive mounted:", os.path.exists("/content/drive/MyDrive"))

print("\nMyDrive folders:")

for item in os.listdir("/content/drive/MyDrive"):
    print(item)

Drive mounted: True

MyDrive folders:
Leela Documents (1).pdf
miniteen.gif
Flare_Removal_Project


In [ ]:
PROJECT = "/content/drive/MyDrive/Flare_Removal_Project"

print("Project folder exists:", os.path.exists(PROJECT))

if os.path.exists(PROJECT):
    for root, dirs, files in os.walk(PROJECT):
        level = root.replace(PROJECT, "").count(os.sep)

        if level <= 3:
            print("  " * level + os.path.basename(root) + "/")

            for file in files:
                print("  " * (level + 1) + file)

Project folder exists: True
Flare_Removal_Project/
  Flare7K_Team_Complete_Code_and_Colab_Notebooks.zip
  data/
    archives/
      Flickr24K.zip
      Flare7K++.zip
    extracted/
    manifests/
  checkpoints/
    baselines/
    improved/
    baseline/
      baseline_unet_recovered.pth
      baseline_unet_latest.pth
      baseline_unet_backup.pth
  logs/
  results/
    metrics/
    figures/


In [ ]:
import os

drive_path = "/content/drive/MyDrive"

print("Google Drive mounted:", os.path.exists(drive_path))

if os.path.exists(drive_path):
    print("\nFolders and files in MyDrive:\n")

    for item in os.listdir(drive_path):
        full_path = os.path.join(drive_path, item)

        if os.path.isdir(full_path):
            print("📁", item)
        else:
            print("📄", item)

Google Drive mounted: True

Folders and files in MyDrive:

📄 Leela Documents (1).pdf
📄 miniteen.gif
📁 Flare_Removal_Project


In [ ]:
project_path = "/content/drive/MyDrive/Flare_Removal_Project"

if os.path.exists(project_path):
    print("Project folder contents:")

    for root, dirs, files in os.walk(project_path):
        level = root.replace(project_path, "").count(os.sep)

        if level <= 3:
            print("  " * level + "📁 " + os.path.basename(root))

            for file in files:
                print("  " * (level + 1) + "📄 " + file)
else:
    print("Flare_Removal_Project folder not found!")

Project folder contents:
📁 Flare_Removal_Project
  📄 Flare7K_Team_Complete_Code_and_Colab_Notebooks.zip
  📁 data
    📁 archives
      📄 Flickr24K.zip
      📄 Flare7K++.zip
    📁 extracted
    📁 manifests
  📁 checkpoints
    📁 baselines
    📁 improved
    📁 baseline
      📄 baseline_unet_recovered.pth
      📄 baseline_unet_latest.pth
      📄 baseline_unet_backup.pth
  📁 logs
  📁 results
    📁 metrics
    📁 figures


In [ ]:
import os
import torch

PROJECT = "/content/drive/MyDrive/Flare_Removal_Project"

checkpoint_dir = os.path.join(
    PROJECT, "checkpoints", "baseline"
)

for filename in [
    "baseline_unet_recovered.pth",
    "baseline_unet_latest.pth",
    "baseline_unet_backup.pth"
]:
    path = os.path.join(checkpoint_dir, filename)

    print("\nFile:", filename)
    print("Exists:", os.path.exists(path))

    if os.path.exists(path):
        print("Size:", round(os.path.getsize(path) / (1024**2), 2), "MB")

        ckpt = torch.load(
            path,
            map_location="cpu",
            weights_only=False
        )

        print("Epoch:", ckpt.get("epoch"))
        print("Batch:", ckpt.get("batch_number"))
        print("Loss:", ckpt.get("loss"))


File: baseline_unet_recovered.pth
Exists: True
Size: 89.01 MB
Epoch: 0
Batch: None
Loss: None

File: baseline_unet_latest.pth
Exists: True
Size: 89.01 MB
Epoch: 0
Batch: 2500
Loss: 0.053835224360227585

File: baseline_unet_backup.pth
Exists: True
Size: 89.01 MB
Epoch: 0
Batch: 2500
Loss: 0.053835224360227585


In [ ]:
print("Model:", "Ready" if "model" in globals() else "Missing")
print("Optimizer:", "Ready" if "optimizer" in globals() else "Missing")
print("Train loader:", "Ready" if "train_loader" in globals() else "Missing")
print("Validation loader:", "Ready" if "val_loader" in globals() else "Missing")
print("GPU:", torch.cuda.is_available())

Model: Missing
Optimizer: Missing
Train loader: Missing
Validation loader: Missing
GPU: False


In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())
else:
    print("GPU is not available yet.")

PyTorch version: 2.14.1+cu130
CUDA available: True
GPU name: Tesla T4
GPU count: 1


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import torch

PROJECT = "/content/drive/MyDrive/Flare_Removal_Project"
CHECKPOINT_DIR = os.path.join(PROJECT, "checkpoints", "baseline")

files = [
    "baseline_unet_recovered.pth",
    "baseline_unet_latest.pth",
    "baseline_unet_backup.pth"
]

for filename in files:
    path = os.path.join(CHECKPOINT_DIR, filename)

    print("\nFile:", filename)
    print("Exists:", os.path.exists(path))

    if os.path.exists(path):
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        print("Epoch:", ckpt.get("epoch"))
        print("Batch:", ckpt.get("batch_number"))
        print("Loss:", ckpt.get("loss"))


File: baseline_unet_recovered.pth
Exists: True
Epoch: 0
Batch: None
Loss: None

File: baseline_unet_latest.pth
Exists: True
Epoch: 0
Batch: 2500
Loss: 0.053835224360227585

File: baseline_unet_backup.pth
Exists: True
Epoch: 0
Batch: 2500
Loss: 0.053835224360227585


In [ ]:
import os
import torch

print("GPU:", torch.cuda.get_device_name(0))

for name in [
    "model",
    "optimizer",
    "train_loader",
    "val_loader",
    "UNet",
    "Flare_Image_Loader"
]:
    print(name, ":", name in globals())

GPU: Tesla T4
model : False
optimizer : False
train_loader : False
val_loader : False
UNet : False
Flare_Image_Loader : False


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import torch
import zipfile

PROJECT = "/content/drive/MyDrive/Flare_Removal_Project"

CHECKPOINT_DIR = os.path.join(PROJECT, "checkpoints", "baseline")
ARCHIVE_DIR = os.path.join(PROJECT, "data", "archives")

checkpoint_path = os.path.join(
    CHECKPOINT_DIR, "baseline_unet_latest.pth"
)

print("GPU:", torch.cuda.get_device_name(0))
print("\nCheckpoint exists:", os.path.exists(checkpoint_path))

print("\nDataset archives:")
for name in ["Flare7K++.zip", "Flickr24K.zip"]:
    path = os.path.join(ARCHIVE_DIR, name)
    print(name, ":", os.path.exists(path))

print("\nCheckpoint details:")
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("Saved epoch:", checkpoint["epoch"])
print("Saved batch:", checkpoint["batch_number"])
print("Saved loss:", checkpoint["loss"])
print("Available keys:", checkpoint.keys())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: Tesla T4

Checkpoint exists: True

Dataset archives:
Flare7K++.zip : True
Flickr24K.zip : True

Checkpoint details:
Saved epoch: 0
Saved batch: 2500
Saved loss: 0.053835224360227585
Available keys: dict_keys(['epoch', 'batch_number', 'model_state_dict', 'optimizer_state_dict', 'loss', 'total_epochs', 'saved_at'])


In [ ]:
import os

paths = {
    "Flare7K++": "/content/flare_data/Flare7Kpp",
    "Flickr24K": "/content/flare_data/Flickr24K",
    "Official Flare7K repository": "/content/Flare7K"
}

for name, path in paths.items():
    print("\n", name)
    print("Path exists:", os.path.exists(path))

    if os.path.exists(path):
        print("Contents:", os.listdir(path)[:10])


 Flare7K++
Path exists: False

 Flickr24K
Path exists: False

 Official Flare7K repository
Path exists: False


In [ ]:
import os
import zipfile

FLARE_ZIP = "/content/drive/MyDrive/Flare_Removal_Project/data/archives/Flare7K++.zip"
FLICKR_ZIP = "/content/drive/MyDrive/Flare_Removal_Project/data/archives/Flickr24K.zip"

os.makedirs("/content/flare_data", exist_ok=True)

datasets = [
    (FLARE_ZIP, "/content/flare_data/Flare7Kpp", "Flare7K++"),
    (FLICKR_ZIP, "/content/flare_data/Flickr24K", "Flickr24K")
]

for zip_path, extract_path, name in datasets:
    print("\nExtracting:", name)

    if not os.path.exists(zip_path):
        print("ERROR: ZIP file not found!")
        continue

    os.makedirs(extract_path, exist_ok=True)

    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_path)

    print(name, "extracted successfully!")
    print("Contents:", os.listdir(extract_path)[:10])

print("\nDataset extraction completed.")


Extracting: Flare7K++
Flare7K++ extracted successfully!
Contents: ['Flare7Kpp']

Extracting: Flickr24K
Flickr24K extracted successfully!
Contents: ['Flickr24K']

Dataset extraction completed.


In [ ]:
import os

BASE = "/content/flare_data"

for root, dirs, files in os.walk(BASE):
    depth = root.replace(BASE, "").count(os.sep)

    if depth <= 3:
        print("\nFolder:", root)
        print("Subfolders:", dirs[:10])
        print("Files:", files[:5])


Folder: /content/flare_data
Subfolders: ['Flare7Kpp', 'Flickr24K']
Files: []

Folder: /content/flare_data/Flare7Kpp
Subfolders: ['Flare7Kpp']
Files: []

Folder: /content/flare_data/Flare7Kpp/Flare7Kpp
Subfolders: ['Flare-R', 'Flare7K', 'test_data']
Files: []

Folder: /content/flare_data/Flare7Kpp/Flare7Kpp/Flare-R
Subfolders: ['Compound_Flare', 'Light_Source']
Files: []

Folder: /content/flare_data/Flare7Kpp/Flare7Kpp/Flare7K
Subfolders: ['Reflective_Flare', 'Scattering_Flare']
Files: []

Folder: /content/flare_data/Flare7Kpp/Flare7Kpp/test_data
Subfolders: ['real', 'synthetic']
Files: []

Folder: /content/flare_data/Flickr24K
Subfolders: ['Flickr24K']
Files: []

Folder: /content/flare_data/Flickr24K/Flickr24K
Subfolders: []
Files: ['r (811).jpg', '11055.jpg', 'r (12419).jpg', 'r (2703).jpg', 'r (10545).jpg']


In [ ]:
import os
import subprocess

OFFICIAL_REPO = "/content/Flare7K"

if not os.path.exists(OFFICIAL_REPO):
    subprocess.run([
        "git", "clone",
        "https://github.com/ykdai/Flare7K",
        OFFICIAL_REPO
    ], check=True)

print("Repository exists:", os.path.exists(OFFICIAL_REPO))
print("data_loader.py exists:",
      os.path.exists(os.path.join(OFFICIAL_REPO, "data_loader.py")))

print("\nRepository files:")
print(os.listdir(OFFICIAL_REPO)[:15])

Repository exists: True
data_loader.py exists: True

Repository files:
['basicsr', 'test.py', 'setup.py', 'LICENSE', 'experiments', 'Generate_flare_on_light', 'test_large.py', 'options', 'VERSION', '.git', 'README.md', 'scripts', 'assets', 'generate_flare.ipynb', 'evaluate.py']


In [ ]:
print("DATA LOADER RESTORATION")
print("Total dataset samples:", len(flare_image_loader))
print("GPU:", torch.cuda.get_device_name(0))

# Inspect available loader attributes
print("\nAvailable loader attributes:")
print([name for name in dir(flare_image_loader)
       if not name.startswith("__")])

DATA LOADER RESTORATION
Total dataset samples: 23949
GPU: Tesla T4

Available loader attributes:
['data_list', 'ext', 'flare_dict', 'flare_list', 'flare_name_list', 'load_reflective_flare', 'load_scattering_flare', 'mask_type', 'reflective_dict', 'reflective_flag', 'reflective_list', 'reflective_name_list', 'transform_base', 'transform_flare']


In [ ]:
from torch.utils.data import DataLoader, random_split

# Dataset split
total_size = len(flare_image_loader)

train_size = int(0.8 * total_size)
val_size = total_size - train_size

generator = torch.Generator().manual_seed(42)

train_dataset, val_dataset = random_split(
    flare_image_loader,
    [train_size, val_size],
    generator=generator
)

# Data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("DATASET SPLIT")
print("Total samples:", total_size)
print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

print("\nBATCH INFORMATION")
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

DATASET SPLIT
Total samples: 23949
Training samples: 19159
Validation samples: 4790

BATCH INFORMATION
Training batches: 4790
Validation batches: 1198


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=3):
        super().__init__()

        self.enc1 = DoubleConv(in_channels, 32)
        self.enc2 = DoubleConv(32, 64)
        self.enc3 = DoubleConv(64, 128)
        self.enc4 = DoubleConv(128, 256)

        self.pool = nn.MaxPool2d(kernel_size=2)

        self.bottleneck = DoubleConv(256, 512)

        self.up4 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(512, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(64, 32)

        self.final = nn.Conv2d(32, out_channels, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.bottleneck(self.pool(e4))

        d4 = self.up4(b)
        d4 = self.dec4(torch.cat([d4, e4], dim=1))

        d3 = self.up3(d4)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))

        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return torch.sigmoid(self.final(d1))


# Initialize model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet().to(device)

# Original loss and optimizer
criterion = nn.L1Loss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

print("U-Net architecture recreated successfully!")
print("Device:", device)
print("Model parameters:", sum(p.numel() for p in model.parameters()))

U-Net architecture recreated successfully!
Device: cuda
Model parameters: 7766051


In [ ]:
import os
import torch

PROJECT = "/content/drive/MyDrive/Flare_Removal_Project"

CHECKPOINT_DIR = os.path.join(
    PROJECT, "checkpoints", "baseline"
)

checkpoint_path = os.path.join(
    CHECKPOINT_DIR, "baseline_unet_latest.pth"
)

# Load saved checkpoint
checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

# Restore model weights
model.load_state_dict(checkpoint["model_state_dict"])

# Restore optimizer state
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

# Recovery information
saved_epoch = checkpoint["epoch"]
saved_batch = checkpoint["batch_number"]
saved_loss = checkpoint["loss"]
EPOCHS = checkpoint["total_epochs"]

print("CHECKPOINT RESTORED SUCCESSFULLY")
print("--------------------------------")
print("Saved epoch:", saved_epoch)
print("Saved batch:", saved_batch)
print("Saved loss:", saved_loss)
print("Total planned epochs:", EPOCHS)
print("Model restored:", True)
print("Optimizer restored:", True)
print("Device:", device)

CHECKPOINT RESTORED SUCCESSFULLY
--------------------------------
Saved epoch: 0
Saved batch: 2500
Saved loss: 0.053835224360227585
Total planned epochs: 20
Model restored: True
Optimizer restored: True
Device: cuda


In [ ]:
import os
import shutil
import torch

CHECKPOINT_DIR = "/content/drive/MyDrive/Flare_Removal_Project/checkpoints/baseline"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

LATEST_PATH = os.path.join(
    CHECKPOINT_DIR, "baseline_unet_latest.pth"
)

BACKUP_PATH = os.path.join(
    CHECKPOINT_DIR, "baseline_unet_backup.pth"
)


def save_checkpoint(epoch, batch_number, loss):

    checkpoint_data = {
        "epoch": epoch,
        "batch_number": batch_number,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "loss": float(loss),
        "total_epochs": EPOCHS,
        "saved_at": str(torch.tensor(0).item())
    }

    temp_latest = LATEST_PATH + ".tmp"
    temp_backup = BACKUP_PATH + ".tmp"

    # Preserve the previous latest checkpoint
    if os.path.exists(LATEST_PATH):
        shutil.copy2(LATEST_PATH, temp_backup)
        os.replace(temp_backup, BACKUP_PATH)

    # Save the new checkpoint safely
    torch.save(checkpoint_data, temp_latest)
    os.replace(temp_latest, LATEST_PATH)

    print(
        f"Checkpoint saved | Epoch: {epoch} | "
        f"Batch: {batch_number} | Loss: {float(loss):.6f}"
    )


print("Automatic checkpoint system is ready!")
print("Latest:", LATEST_PATH)
print("Backup:", BACKUP_PATH)
print("Save interval: Every 100 batches")

Automatic checkpoint system is ready!
Latest: /content/drive/MyDrive/Flare_Removal_Project/checkpoints/baseline/baseline_unet_latest.pth
Backup: /content/drive/MyDrive/Flare_Removal_Project/checkpoints/baseline/baseline_unet_backup.pth
Save interval: Every 100 batches


In [ ]:
import torch
import time

start_epoch = 1  # Epoch 0 was partially completed
EPOCHS = 20

for epoch in range(start_epoch, EPOCHS):

    model.train()
    running_loss = 0.0

    start_time = time.time()

    print(f"\n========== EPOCH {epoch + 1}/{EPOCHS} ==========")

    for batch_idx, batch in enumerate(train_loader):

        # Original loader returns five items
        base_img, flare_img, merge_img, flare_mask, gamma = batch

        inputs = merge_img.to(device)
        targets = base_img.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        current_batch = batch_idx + 1

        # Save every 100 batches
        if current_batch % 100 == 0:
            save_checkpoint(
                epoch,
                current_batch,
                loss.item()
            )

        if current_batch % 100 == 0:
            print(
                f"Epoch [{epoch + 1}/{EPOCHS}] "
                f"Batch [{current_batch}/{len(train_loader)}] "
                f"Loss: {loss.item():.6f}"
            )

    epoch_loss = running_loss / len(train_loader)

    # Save completed epoch
    save_checkpoint(epoch + 1, 0, epoch_loss)

    elapsed = (time.time() - start_time) / 60

    print("\nEpoch completed!")
    print(f"Average training loss: {epoch_loss:.6f}")
    print(f"Time taken: {elapsed:.2f} minutes")


========== EPOCH 2/20 ==========
Checkpoint saved | Epoch: 1 | Batch: 100 | Loss: 0.067152
Epoch [2/20] Batch [100/4790] Loss: 0.067152
Checkpoint saved | Epoch: 1 | Batch: 200 | Loss: 0.053753
Epoch [2/20] Batch [200/4790] Loss: 0.053753
Checkpoint saved | Epoch: 1 | Batch: 300 | Loss: 0.095138
Epoch [2/20] Batch [300/4790] Loss: 0.095138
Checkpoint saved | Epoch: 1 | Batch: 400 | Loss: 0.043468
Epoch [2/20] Batch [400/4790] Loss: 0.043468
Checkpoint saved | Epoch: 1 | Batch: 500 | Loss: 0.059626
Epoch [2/20] Batch [500/4790] Loss: 0.059626
Checkpoint saved | Epoch: 1 | Batch: 600 | Loss: 0.040348
Epoch [2/20] Batch [600/4790] Loss: 0.040348
Checkpoint saved | Epoch: 1 | Batch: 700 | Loss: 0.050707
Epoch [2/20] Batch [700/4790] Loss: 0.050707
Checkpoint saved | Epoch: 1 | Batch: 800 | Loss: 0.081094
Epoch [2/20] Batch [800/4790] Loss: 0.081094
Checkpoint saved | Epoch: 1 | Batch: 900 | Loss: 0.091782
Epoch [2/20] Batch [900/4790] Loss: 0.091782
Checkpoint saved | Epoch: 1 | Batch: 10

KeyboardInterrupt: 

In [ ]:
import os
import torch

path = "/content/drive/MyDrive/Flare_Removal_Project/checkpoints/baseline/baseline_unet_latest.pth"

checkpoint = torch.load(
    path,
    map_location="cpu",
    weights_only=False
)

print("CHECKPOINT VERIFICATION")
print("Epoch:", checkpoint["epoch"])
print("Batch:", checkpoint["batch_number"])
print("Loss:", checkpoint["loss"])
print("File size (MB):", round(os.path.getsize(path) / (1024**2), 2))
print("Model weights:", "model_state_dict" in checkpoint)
print("Optimizer state:", "optimizer_state_dict" in checkpoint)

CHECKPOINT VERIFICATION
Epoch: 1
Batch: 2500
Loss: 0.048331305384635925
File size (MB): 89.01
Model weights: True
Optimizer state: True


In [ ]:
import torch
import os

print("GPU available:", torch.cuda.is_available())
print("Model available:", 'model' in globals())
print("Dataset available:", 'flare_image_loader' in globals())
print("Drive available:", os.path.exists(
    "/content/drive/MyDrive/Flare_Removal_Project"
))

GPU available: True
Model available: True
Dataset available: True
Drive available: True


In [ ]:
import os
import sys
import torch
from torchvision import transforms
from torch.utils.data import random_split, DataLoader, Subset

sys.path.append("/content/Flare7K")
from data_loader import Flare_Image_Loader

# Dataset paths
FLARE_ROOT = "/content/flare_data/Flare7Kpp/Flare7Kpp"
FLICKR_ROOT = "/content/flare_data/Flickr24K/Flickr24K"

# Transform for clean base images
transform_base_256 = transforms.Compose([
    transforms.RandomCrop(
        (256, 256),
        pad_if_needed=True,
        padding_mode="reflect"
    ),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip()
])

# Transform for flare patterns
transform_flare_256 = transforms.Compose([
    transforms.RandomAffine(
        degrees=(0, 360),
        scale=(0.8, 1.5),
        translate=(300/1440, 300/1440),
        shear=(-20, 20)
    ),
    transforms.CenterCrop((256, 256)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip()
])

# Create a separate 256x256 dataset loader
flare_image_loader_256 = Flare_Image_Loader(
    FLICKR_ROOT,
    transform_base_256,
    transform_flare_256,
    mask_type="luminance"
)

# Load scattering flare patterns
flare_image_loader_256.load_scattering_flare(
    "Flare7K",
    os.path.join(
        FLARE_ROOT,
        "Flare7K",
        "Scattering_Flare",
        "Compound_Flare"
    )
)

# Load reflective flare patterns
flare_image_loader_256.load_reflective_flare(
    "Flare7K",
    os.path.join(
        FLARE_ROOT,
        "Flare7K",
        "Reflective_Flare"
    )
)

print("256x256 dataset created successfully!")
print("Total images:", len(flare_image_loader_256))

# Check one sample
base_img, flare_img, merge_img, flare_mask, gamma = flare_image_loader_256[0]

print("Base image shape:", base_img.shape)
print("Flare image shape:", flare_img.shape)
print("Merged image shape:", merge_img.shape)
print("Flare mask shape:", flare_mask.shape)

Base Image Loaded with examples: 23949
Scattering Flare Image: Flare7K  is loaded successfully with examples 5000
Now we have 5000 scattering flare images
Reflective Flare Image: Flare7K  is loaded successfully with examples 2000
Now we have 2000 refelctive flare images
256x256 dataset created successfully!
Total images: 23949
Base image shape: torch.Size([3, 256, 256])
Flare image shape: torch.Size([3, 256, 256])
Merged image shape: torch.Size([3, 256, 256])
Flare mask shape: torch.Size([3, 256, 256])


In [ ]:
from torch.utils.data import random_split, DataLoader, Subset

# Split the complete dataset (80% training, 20% validation)
total_size = len(flare_image_loader_256)

train_size = int(0.8 * total_size)
val_size = total_size - train_size

generator = torch.Generator().manual_seed(42)

train_dataset_256, val_dataset_256 = random_split(
    flare_image_loader_256,
    [train_size, val_size],
    generator=generator
)

# Select smaller subsets for balanced training
train_dataset_256 = Subset(
    train_dataset_256,
    range(10000)
)

val_dataset_256 = Subset(
    val_dataset_256,
    range(2500)
)

# Create DataLoaders
BATCH_SIZE = 8

train_loader_256 = DataLoader(
    train_dataset_256,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader_256 = DataLoader(
    val_dataset_256,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Dataset preparation completed!")
print("Training images:", len(train_dataset_256))
print("Validation images:", len(val_dataset_256))
print("Training batches:", len(train_loader_256))
print("Validation batches:", len(val_loader_256))
print("Batch size:", BATCH_SIZE)

Dataset preparation completed!
Training images: 10000
Validation images: 2500
Training batches: 1250
Validation batches: 313
Batch size: 8


In [ ]:
import torch
import torch.optim as optim
import os

# Original checkpoint from 512x512 training
OLD_CHECKPOINT_PATH = (
    "/content/drive/MyDrive/Flare_Removal_Project/"
    "checkpoints/baseline/baseline_unet_latest.pth"
)

# Load saved checkpoint
checkpoint = torch.load(
    OLD_CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)

# Restore learned model weights
model.load_state_dict(checkpoint["model_state_dict"])

# Move model to GPU
model = model.to(device)

# Create fresh optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=0.0001
)

# Loss function
criterion = torch.nn.L1Loss()

# New training configuration
EPOCHS = 10
BATCH_SIZE = 8
IMAGE_SIZE = 256

print("Previous checkpoint restored successfully!")
print("Previous epoch:", checkpoint["epoch"])
print("Previous batch:", checkpoint["batch_number"])
print("Previous loss:", checkpoint["loss"])

print("\nNew training configuration")
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)
print("Device:", device)
print("Model parameters:", sum(p.numel() for p in model.parameters()))

Previous checkpoint restored successfully!
Previous epoch: 1
Previous batch: 2500
Previous loss: 0.048331305384635925

New training configuration
Image size: 256
Batch size: 8
Epochs: 10
Device: cuda
Model parameters: 7766051


In [ ]:
# Enable mixed precision training
scaler = torch.amp.GradScaler("cuda")

print("Mixed precision configured!")
print("Precision: FP16")
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory:", round(
    torch.cuda.get_device_properties(0).total_memory / (1024**3), 2
), "GB")
print("GradScaler enabled:", scaler.is_enabled())

Mixed precision configured!
Precision: FP16
GPU: Tesla T4
GPU memory: 14.56 GB
GradScaler enabled: True


In [ ]:
import torch

model.train()

# Get one batch
base_img, flare_img, merge_img, flare_mask, gamma = next(iter(train_loader_256))

inputs = merge_img.to(device, non_blocking=True)
targets = base_img.to(device, non_blocking=True)

# Clear gradients
optimizer.zero_grad(set_to_none=True)

# Mixed precision forward pass
with torch.autocast(device_type="cuda", dtype=torch.float16):
    outputs = model(inputs)
    loss = criterion(outputs.float(), targets.float())

# Backpropagation with gradient scaling
scaler.scale(loss).backward()
scaler.step(optimizer)
scaler.update()

torch.cuda.synchronize()

print("Training batch test successful!")
print("Input shape:", inputs.shape)
print("Output shape:", outputs.shape)
print("Batch size:", inputs.shape[0])
print("Loss:", loss.item())
print("GPU memory used:", round(
    torch.cuda.max_memory_allocated() / (1024**3), 2
), "GB")

Training batch test successful!
Input shape: torch.Size([8, 3, 256, 256])
Output shape: torch.Size([8, 3, 256, 256])
Batch size: 8
Loss: 0.047163404524326324
GPU memory used: 3.42 GB


In [ ]:
import os
import torch
import shutil
from datetime import datetime

# Separate folder for optimized training
CHECKPOINT_DIR = (
    "/content/drive/MyDrive/Flare_Removal_Project/"
    "checkpoints/baseline_256"
)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

LATEST_PATH = os.path.join(
    CHECKPOINT_DIR, "baseline_unet_256_latest.pth"
)

BACKUP_PATH = os.path.join(
    CHECKPOINT_DIR, "baseline_unet_256_backup.pth"
)

def save_checkpoint(epoch, batch_number, current_loss):

    checkpoint_data = {
        "epoch": epoch,
        "batch_number": batch_number,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "loss": float(current_loss),
        "total_epochs": EPOCHS,
        "image_size": IMAGE_SIZE,
        "batch_size": BATCH_SIZE,
        "train_samples": len(train_dataset_256),
        "val_samples": len(val_dataset_256),
        "saved_at": datetime.now().isoformat()
    }

    temp_latest = LATEST_PATH + ".tmp"
    temp_backup = BACKUP_PATH + ".tmp"

    # Preserve the previous latest checkpoint
    if os.path.exists(LATEST_PATH):
        shutil.copy2(LATEST_PATH, temp_backup)
        os.replace(temp_backup, BACKUP_PATH)

    # Save safely using a temporary file
    torch.save(checkpoint_data, temp_latest)
    os.replace(temp_latest, LATEST_PATH)

    print(
        f"Checkpoint saved | Epoch: {epoch + 1} | "
        f"Batch: {batch_number} | Loss: {current_loss:.6f}"
    )


# Save the successful test update immediately
save_checkpoint(0, 1, loss.item())

print("\nCheckpoint system ready!")
print("Latest:", LATEST_PATH)
print("Backup:", BACKUP_PATH)

Checkpoint saved | Epoch: 1 | Batch: 1 | Loss: 0.047163

Checkpoint system ready!
Latest: /content/drive/MyDrive/Flare_Removal_Project/checkpoints/baseline_256/baseline_unet_256_latest.pth
Backup: /content/drive/MyDrive/Flare_Removal_Project/checkpoints/baseline_256/baseline_unet_256_backup.pth


In [ ]:
import torch
import time

EPOCHS = 10

for epoch in range(EPOCHS):

    print(f"\n========== EPOCH {epoch + 1}/{EPOCHS} ==========")

    # ---------------- TRAINING ----------------
    model.train()
    running_loss = 0.0

    start_time = time.time()

    for batch_idx, batch in enumerate(train_loader_256):

        base_img, flare_img, merge_img, flare_mask, gamma = batch

        inputs = merge_img.to(device, non_blocking=True)
        targets = base_img.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        # FP16 forward pass
        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            outputs = model(inputs)
            loss = criterion(outputs.float(), targets.float())

        # Backpropagation
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

        current_batch = batch_idx + 1

        # Save every 100 batches
        if current_batch % 100 == 0:
            save_checkpoint(
                epoch,
                current_batch,
                loss.item()
            )

            print(
                f"Epoch [{epoch + 1}/{EPOCHS}] "
                f"Batch [{current_batch}/{len(train_loader_256)}] "
                f"Loss: {loss.item():.6f}"
            )

    # Average training loss
    epoch_loss = running_loss / len(train_loader_256)

    # ---------------- VALIDATION ----------------
    model.eval()
    val_loss = 0.0

    with torch.no_grad():

        for batch in val_loader_256:

            base_img, flare_img, merge_img, flare_mask, gamma = batch

            inputs = merge_img.to(device, non_blocking=True)
            targets = base_img.to(device, non_blocking=True)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                outputs = model(inputs)
                loss = criterion(outputs.float(), targets.float())

            val_loss += loss.item()

    val_loss = val_loss / len(val_loader_256)

    # Save at the end of the epoch
    save_checkpoint(
        epoch,
        len(train_loader_256),
        epoch_loss
    )

    elapsed = (time.time() - start_time) / 60

    print("\nEpoch completed!")
    print(f"Average Training Loss: {epoch_loss:.6f}")
    print(f"Average Validation Loss: {val_loss:.6f}")
    print(f"Time taken: {elapsed:.2f} minutes")

print("\n========== TRAINING COMPLETED ==========")


========== EPOCH 1/10 ==========
Checkpoint saved | Epoch: 1 | Batch: 100 | Loss: 0.051825
Epoch [1/10] Batch [100/1250] Loss: 0.051825
Checkpoint saved | Epoch: 1 | Batch: 200 | Loss: 0.072392
Epoch [1/10] Batch [200/1250] Loss: 0.072392
Checkpoint saved | Epoch: 1 | Batch: 300 | Loss: 0.062896
Epoch [1/10] Batch [300/1250] Loss: 0.062896
Checkpoint saved | Epoch: 1 | Batch: 400 | Loss: 0.061906
Epoch [1/10] Batch [400/1250] Loss: 0.061906
Checkpoint saved | Epoch: 1 | Batch: 500 | Loss: 0.046929
Epoch [1/10] Batch [500/1250] Loss: 0.046929
Checkpoint saved | Epoch: 1 | Batch: 600 | Loss: 0.050858
Epoch [1/10] Batch [600/1250] Loss: 0.050858
Checkpoint saved | Epoch: 1 | Batch: 700 | Loss: 0.051801
Epoch [1/10] Batch [700/1250] Loss: 0.051801
Checkpoint saved | Epoch: 1 | Batch: 800 | Loss: 0.040824
Epoch [1/10] Batch [800/1250] Loss: 0.040824
Checkpoint saved | Epoch: 1 | Batch: 900 | Loss: 0.042040
Epoch [1/10] Batch [900/1250] Loss: 0.042040
Checkpoint saved | Epoch: 1 | Batch: 10

KeyboardInterrupt: 

In [ ]:
import torch
import os

path = "/content/drive/MyDrive/Flare_Removal_Project/checkpoints/baseline_256/baseline_unet_256_latest.pth"

ckpt = torch.load(path, map_location="cpu", weights_only=False)

print("CHECKPOINT VERIFICATION")
print("Epoch:", ckpt["epoch"] + 1)
print("Batch:", ckpt["batch_number"])
print("Loss:", ckpt["loss"])
print("Image size:", ckpt["image_size"])
print("Model weights:", "model_state_dict" in ckpt)
print("Optimizer state:", "optimizer_state_dict" in ckpt)
print("Scaler state:", "scaler_state_dict" in ckpt)
print("File size (MB):", round(os.path.getsize(path) / (1024**2), 2))

CHECKPOINT VERIFICATION
Epoch: 1
Batch: 1000
Loss: 0.05507345125079155
Image size: 256
Model weights: True
Optimizer state: True
Scaler state: True
File size (MB): 89.02


In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("GPU memory:",
          round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2),
          "GB")

GPU available: True
GPU name: Tesla T4
GPU memory: 14.56 GB


In [ ]:
import os

checkpoint_path = (
    "/content/drive/MyDrive/Flare_Removal_Project/"
    "checkpoints/baseline_256/baseline_unet_256_latest.pth"
)

print("Drive mounted:", os.path.exists("/content/drive/MyDrive"))
print("Checkpoint found:", os.path.exists(checkpoint_path))

Drive mounted: True
Checkpoint found: True


In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/Flare_Removal_Project"

paths = {
    "Flare7K++ archive": os.path.join(
        PROJECT_DIR, "data/archives/Flare7K++.zip"
    ),
    "Flickr24K archive": os.path.join(
        PROJECT_DIR, "data/archives/Flickr24K.zip"
    ),
    "Flare7K++ extracted": "/content/flare_data/Flare7Kpp/Flare7Kpp",
    "Flickr24K extracted": "/content/flare_data/Flickr24K/Flickr24K",
    "Official repository": "/content/Flare7K"
}

for name, path in paths.items():
    print(name, ":", os.path.exists(path))

Flare7K++ archive : True
Flickr24K archive : True
Flare7K++ extracted : False
Flickr24K extracted : False
Official repository : False


In [ ]:
import os
import zipfile
import subprocess

PROJECT_DIR = "/content/drive/MyDrive/Flare_Removal_Project"

# Create extraction directories
os.makedirs("/content/flare_data/Flare7Kpp", exist_ok=True)
os.makedirs("/content/flare_data/Flickr24K", exist_ok=True)

# Extract Flare7K++
flare_zip = os.path.join(
    PROJECT_DIR, "data/archives/Flare7K++.zip"
)

print("Extracting Flare7K++...")
with zipfile.ZipFile(flare_zip, "r") as zip_ref:
    zip_ref.extractall("/content/flare_data/Flare7Kpp")

print("Flare7K++ extraction completed!")

# Extract Flickr24K
flickr_zip = os.path.join(
    PROJECT_DIR, "data/archives/Flickr24K.zip"
)

print("\nExtracting Flickr24K...")
with zipfile.ZipFile(flickr_zip, "r") as zip_ref:
    zip_ref.extractall("/content/flare_data/Flickr24K")

print("Flickr24K extraction completed!")

# Clone official repository
print("\nCloning official Flare7K repository...")

subprocess.run([
    "git", "clone",
    "https://github.com/ykdai/Flare7K",
    "/content/Flare7K"
], check=True)

print("\n========== RESTORATION CHECK ==========")

print("Flare7K++:",
      os.path.exists("/content/flare_data/Flare7Kpp/Flare7Kpp"))

print("Flickr24K:",
      os.path.exists("/content/flare_data/Flickr24K/Flickr24K"))

print("Official loader:",
      os.path.exists("/content/Flare7K/data_loader.py"))

print("======================================")

Extracting Flare7K++...
Flare7K++ extraction completed!

Extracting Flickr24K...
Flickr24K extraction completed!

Cloning official Flare7K repository...

========== RESTORATION CHECK ==========
Flare7K++: True
Flickr24K: True
Official loader: True


In [ ]:
import os
import sys
import torch
from torchvision import transforms
from torch.utils.data import random_split, DataLoader, Subset

# Import official Flare7K loader
sys.path.append("/content/Flare7K")
from data_loader import Flare_Image_Loader

# Dataset paths
FLARE_ROOT = "/content/flare_data/Flare7Kpp/Flare7Kpp"
FLICKR_ROOT = "/content/flare_data/Flickr24K/Flickr24K"

# 256x256 base image transformation
transform_base_256 = transforms.Compose([
    transforms.RandomCrop(
        (256, 256),
        pad_if_needed=True,
        padding_mode="reflect"
    ),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip()
])

# 256x256 flare transformation
transform_flare_256 = transforms.Compose([
    transforms.RandomAffine(
        degrees=(0, 360),
        scale=(0.8, 1.5),
        translate=(300/1440, 300/1440),
        shear=(-20, 20)
    ),
    transforms.CenterCrop((256, 256)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip()
])

# Create dataset loader
flare_image_loader_256 = Flare_Image_Loader(
    FLICKR_ROOT,
    transform_base_256,
    transform_flare_256,
    mask_type="luminance"
)

# Load scattering flare patterns
flare_image_loader_256.load_scattering_flare(
    "Flare7K",
    os.path.join(
        FLARE_ROOT,
        "Flare7K",
        "Scattering_Flare",
        "Compound_Flare"
    )
)

# Load reflective flare patterns
flare_image_loader_256.load_reflective_flare(
    "Flare7K",
    os.path.join(
        FLARE_ROOT,
        "Flare7K",
        "Reflective_Flare"
    )
)

# Verify dataset
print("\nDataset restoration successful!")
print("Total images:", len(flare_image_loader_256))

base_img, flare_img, merge_img, flare_mask, gamma = flare_image_loader_256[0]

print("Base:", base_img.shape)
print("Flare:", flare_img.shape)
print("Merged:", merge_img.shape)
print("Mask:", flare_mask.shape)

Base Image Loaded with examples: 23949
Scattering Flare Image: Flare7K  is loaded successfully with examples 5000
Now we have 5000 scattering flare images
Reflective Flare Image: Flare7K  is loaded successfully with examples 2000
Now we have 2000 refelctive flare images

Dataset restoration successful!
Total images: 23949
Base: torch.Size([3, 256, 256])
Flare: torch.Size([3, 256, 256])
Merged: torch.Size([3, 256, 256])
Mask: torch.Size([3, 256, 256])


In [ ]:
from torch.utils.data import random_split, DataLoader, Subset

# Recreate original 80:20 split
total_size = len(flare_image_loader_256)

train_size = int(0.8 * total_size)
val_size = total_size - train_size

generator = torch.Generator().manual_seed(42)

train_dataset_256, val_dataset_256 = random_split(
    flare_image_loader_256,
    [train_size, val_size],
    generator=generator
)

# Select the same training and validation subsets
train_dataset_256 = Subset(train_dataset_256, range(10000))
val_dataset_256 = Subset(val_dataset_256, range(2500))

# Recreate DataLoaders
BATCH_SIZE = 8

train_loader_256 = DataLoader(
    train_dataset_256,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader_256 = DataLoader(
    val_dataset_256,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Training images:", len(train_dataset_256))
print("Validation images:", len(val_dataset_256))
print("Training batches:", len(train_loader_256))
print("Validation batches:", len(val_loader_256))
print("Batch size:", BATCH_SIZE)

Training images: 10000
Validation images: 2500
Training batches: 1250
Validation batches: 313
Batch size: 8


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Double Convolution block
class DoubleConv(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


# U-Net architecture
class UNet(nn.Module):

    def __init__(self, in_channels=3, out_channels=3):
        super().__init__()

        self.enc1 = DoubleConv(in_channels, 32)
        self.enc2 = DoubleConv(32, 64)
        self.enc3 = DoubleConv(64, 128)
        self.enc4 = DoubleConv(128, 256)

        self.pool = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(256, 512)

        self.up4 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec4 = DoubleConv(512, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec3 = DoubleConv(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = DoubleConv(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = DoubleConv(64, 32)

        self.final = nn.Conv2d(32, out_channels, 1)

    def forward(self, x):

        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.bottleneck(self.pool(e4))

        d4 = self.up4(b)
        d4 = self.dec4(torch.cat([d4, e4], dim=1))

        d3 = self.up3(d4)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))

        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return torch.sigmoid(self.final(d1))


# Initialize model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet().to(device)

print("U-Net architecture recreated!")
print("Device:", device)
print("Parameters:", sum(p.numel() for p in model.parameters()))

U-Net architecture recreated!
Device: cuda
Parameters: 7766051


In [ ]:
checkpoint_path = (
    "/content/drive/MyDrive/Flare_Removal_Project/"
    "checkpoints/baseline_256/baseline_unet_256_latest.pth"
)

# Initialize training components
criterion = nn.L1Loss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)
scaler = torch.amp.GradScaler("cuda")

# Load saved checkpoint
checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
scaler.load_state_dict(checkpoint["scaler_state_dict"])

# Restore training position
saved_epoch = checkpoint["epoch"]
saved_batch = checkpoint["batch_number"]

print("CHECKPOINT RESTORED SUCCESSFULLY")
print("Completed epoch:", saved_epoch + 1)
print("Last saved batch:", saved_batch)
print("Last saved loss:", checkpoint["loss"])
print("Image size:", checkpoint["image_size"])
print("Batch size:", checkpoint["batch_size"])
print("Model weights: Restored")
print("Optimizer: Restored")
print("FP16 scaler: Restored")

CHECKPOINT RESTORED SUCCESSFULLY
Completed epoch: 1
Last saved batch: 1000
Last saved loss: 0.05507345125079155
Image size: 256
Batch size: 8
Model weights: Restored
Optimizer: Restored
FP16 scaler: Restored


In [ ]:
import os
import torch

checkpoint_dir = (
    "/content/drive/MyDrive/Flare_Removal_Project/"
    "checkpoints/baseline_256"
)

os.makedirs(checkpoint_dir, exist_ok=True)

latest_path = os.path.join(
    checkpoint_dir, "baseline_unet_256_latest.pth"
)

backup_path = os.path.join(
    checkpoint_dir, "baseline_unet_256_backup.pth"
)

def save_checkpoint(epoch, batch_number, loss):

    checkpoint_data = {
        "epoch": epoch,
        "batch_number": batch_number,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "loss": loss,
        "total_epochs": 10,
        "image_size": 256,
        "batch_size": 8,
        "train_samples": 10000,
        "val_samples": 2500
    }

    temp_path = latest_path + ".tmp"

    torch.save(checkpoint_data, temp_path)

    if os.path.exists(latest_path):
        os.replace(latest_path, backup_path)

    os.replace(temp_path, latest_path)

    print(
        f"Checkpoint saved | Epoch: {epoch+1} "
        f"| Batch: {batch_number} | Loss: {loss:.6f}"
    )

print("Checkpoint saving configured!")
print("Latest:", latest_path)
print("Backup:", backup_path)

Checkpoint saving configured!
Latest: /content/drive/MyDrive/Flare_Removal_Project/checkpoints/baseline_256/baseline_unet_256_latest.pth
Backup: /content/drive/MyDrive/Flare_Removal_Project/checkpoints/baseline_256/baseline_unet_256_backup.pth


In [ ]:
EPOCHS = 10

for epoch in range(saved_epoch, EPOCHS):

    model.train()
    running_loss = 0.0
    processed_batches = 0

    # Use only remaining data for the first resumed epoch
    if epoch == saved_epoch:
        current_loader = remaining_loader
        batch_offset = saved_batch
    else:
        current_loader = train_loader_256
        batch_offset = 0

    print(f"\n========== EPOCH {epoch+1}/{EPOCHS} ==========")

    for batch_idx, batch in enumerate(current_loader):

        inputs = batch[2].to(device, non_blocking=True)
        targets = batch[0].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            outputs = model(inputs)
            loss = criterion(outputs.float(), targets.float())

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        processed_batches += 1

        current_batch = batch_offset + batch_idx + 1

        if current_batch % 100 == 0:
            print(
                f"Epoch [{epoch+1}/{EPOCHS}] "
                f"Batch [{current_batch}/1250] "
                f"Loss: {loss.item():.6f}"
            )

            save_checkpoint(epoch, current_batch, loss.item())

    # Validation
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for batch in val_loader_256:

            inputs = batch[2].to(device, non_blocking=True)
            targets = batch[0].to(device, non_blocking=True)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                outputs = model(inputs)
                loss = criterion(outputs.float(), targets.float())

            val_loss += loss.item()

    val_loss /= len(val_loader_256)
    train_loss = running_loss / processed_batches

    print(f"\nEpoch {epoch+1} completed!")
    print(f"Training Loss: {train_loss:.6f}")
    print(f"Validation Loss: {val_loss:.6f}")

    save_checkpoint(epoch, 1250, train_loss)

print("\n========== TRAINING COMPLETED ==========")


========== EPOCH 1/10 ==========
Epoch [1/10] Batch [1100/1250] Loss: 0.048864
Checkpoint saved | Epoch: 1 | Batch: 1100 | Loss: 0.048864
Epoch [1/10] Batch [1200/1250] Loss: 0.073200
Checkpoint saved | Epoch: 1 | Batch: 1200 | Loss: 0.073200

Epoch 1 completed!
Training Loss: 0.053698
Validation Loss: 0.046538
Checkpoint saved | Epoch: 1 | Batch: 1250 | Loss: 0.053698

========== EPOCH 2/10 ==========
Epoch [2/10] Batch [100/1250] Loss: 0.046524
Checkpoint saved | Epoch: 2 | Batch: 100 | Loss: 0.046524
Epoch [2/10] Batch [200/1250] Loss: 0.061057
Checkpoint saved | Epoch: 2 | Batch: 200 | Loss: 0.061057
Epoch [2/10] Batch [300/1250] Loss: 0.033907
Checkpoint saved | Epoch: 2 | Batch: 300 | Loss: 0.033907
Epoch [2/10] Batch [400/1250] Loss: 0.052555
Checkpoint saved | Epoch: 2 | Batch: 400 | Loss: 0.052555
Epoch [2/10] Batch [500/1250] Loss: 0.069022
Checkpoint saved | Epoch: 2 | Batch: 500 | Loss: 0.069022
Epoch [2/10] Batch [600/1250] Loss: 0.055779
Checkpoint saved | Epoch: 2 | Bat

In [ ]:
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory allocated:",
      round(torch.cuda.memory_allocated()/1024**3, 2), "GB")
print("GPU memory reserved:",
      round(torch.cuda.memory_reserved()/1024**3, 2), "GB")

GPU: Tesla T4
GPU memory allocated: 0.12 GB
GPU memory reserved: 0.13 GB


In [ ]:
checkpoint = torch.load(
    latest_path,
    map_location="cpu",
    weights_only=False
)

print("CHECKPOINT STATUS")
print("Epoch:", checkpoint["epoch"] + 1)
print("Batch:", checkpoint["batch_number"])
print("Loss:", checkpoint["loss"])
print("Image size:", checkpoint["image_size"])
print("Model weights:", "model_state_dict" in checkpoint)
print("Optimizer:", "optimizer_state_dict" in checkpoint)
print("Scaler:", "scaler_state_dict" in checkpoint)

CHECKPOINT STATUS
Epoch: 1
Batch: 1000
Loss: 0.05507345125079155
Image size: 256
Model weights: True
Optimizer: True
Scaler: True


In [ ]:
from torch.utils.data import Subset, DataLoader

# Select only the remaining 250 batches (2000 images)
remaining_indices = train_dataset_256.indices[8000:]

remaining_dataset = Subset(
    train_dataset_256.dataset,
    remaining_indices
)

remaining_loader = DataLoader(
    remaining_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

print("Remaining training images:", len(remaining_dataset))
print("Remaining batches:", len(remaining_loader))

Remaining training images: 2000
Remaining batches: 250


In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("GPU memory:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")
else:
    print("GPU is not available. Do not start training.")

GPU available: True
GPU name: Tesla T4
GPU memory: 14.56 GB


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_DIR = "/content/drive/MyDrive/Flare_Removal_Project"

CHECKPOINT_DIR = os.path.join(
    PROJECT_DIR, "checkpoints", "baseline_256"
)

print("Drive mounted:", os.path.exists(PROJECT_DIR))
print("Latest checkpoint exists:",
      os.path.exists(os.path.join(
          CHECKPOINT_DIR, "baseline_unet_256_latest.pth"
      )))
print("Backup checkpoint exists:",
      os.path.exists(os.path.join(
          CHECKPOINT_DIR, "baseline_unet_256_backup.pth"
      )))

Mounted at /content/drive
Drive mounted: True
Latest checkpoint exists: True
Backup checkpoint exists: True


In [ ]:
import torch
import os

checkpoint_path = os.path.join(
    CHECKPOINT_DIR,
    "baseline_unet_256_latest.pth"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("========== CHECKPOINT VERIFICATION ==========")
print("Completed epoch:", checkpoint["epoch"] + 1)
print("Last saved batch:", checkpoint["batch_number"])
print("Last saved loss:", checkpoint["loss"])
print("Image size:", checkpoint["image_size"])
print("Batch size:", checkpoint["batch_size"])
print("Model weights:", bool(checkpoint["model_state_dict"]))
print("Optimizer:", bool(checkpoint["optimizer_state_dict"]))
print("FP16 scaler:", bool(checkpoint["scaler_state_dict"]))
print("=============================================")

========== CHECKPOINT VERIFICATION ==========
Completed epoch: 4
Last saved batch: 400
Last saved loss: 0.047095887362957
Image size: 256
Batch size: 8
Model weights: True
Optimizer: True
FP16 scaler: True


In [ ]:
import os
import zipfile
import subprocess

PROJECT_DIR = "/content/drive/MyDrive/Flare_Removal_Project"

ARCHIVE_DIR = os.path.join(PROJECT_DIR, "data", "archives")

FLARE_ARCHIVE = os.path.join(ARCHIVE_DIR, "Flare7K++.zip")
FLICKR_ARCHIVE = os.path.join(ARCHIVE_DIR, "Flickr24K.zip")

os.makedirs("/content/flare_data", exist_ok=True)

# Extract Flare7K++
if not os.path.exists("/content/flare_data/Flare7Kpp"):
    with zipfile.ZipFile(FLARE_ARCHIVE, "r") as z:
        z.extractall("/content/flare_data")

# Extract Flickr24K
if not os.path.exists("/content/flare_data/Flickr24K"):
    with zipfile.ZipFile(FLICKR_ARCHIVE, "r") as z:
        z.extractall("/content/flare_data")

# Clone official repository
if not os.path.exists("/content/Flare7K/data_loader.py"):
    subprocess.run(
        ["git", "clone",
         "https://github.com/ykdai/Flare7K",
         "/content/Flare7K"],
        check=True
    )

FLARE_ROOT = "/content/flare_data/Flare7Kpp/Flare7Kpp"
FLICKR_ROOT = "/content/flare_data/Flickr24K/Flickr24K"

print("Flare7K++:", os.path.exists(FLARE_ROOT))
print("Flickr24K:", os.path.exists(FLICKR_ROOT))
print("Official loader:",
      os.path.exists("/content/Flare7K/data_loader.py"))

Flare7K++: False
Flickr24K: False
Official loader: True


In [ ]:
import os
import zipfile

print("========== CONTENTS OF flare_data ==========")

for item in os.listdir("/content/flare_data"):
    print(item)

print("\n========== ARCHIVE CONTENTS ==========")

for archive in [FLARE_ARCHIVE, FLICKR_ARCHIVE]:
    print("\nArchive:", os.path.basename(archive))
    print("Exists:", os.path.exists(archive))

    if os.path.exists(archive):
        with zipfile.ZipFile(archive, "r") as z:
            print("First 10 entries:")
            for name in z.namelist()[:10]:
                print(name)

========== CONTENTS OF flare_data ==========
Flickr24K
Flare7Kpp

========== ARCHIVE CONTENTS ==========

Archive: Flare7K++.zip
Exists: True
First 10 entries:
Flare7Kpp/
Flare7Kpp/Flare-R/
Flare7Kpp/Flare-R/Compound_Flare/
Flare7Kpp/Flare-R/Compound_Flare/000000.png
Flare7Kpp/Flare-R/Compound_Flare/000001.png
Flare7Kpp/Flare-R/Compound_Flare/000002.png
Flare7Kpp/Flare-R/Compound_Flare/000003.png
Flare7Kpp/Flare-R/Compound_Flare/000004.png
Flare7Kpp/Flare-R/Compound_Flare/000005.png
Flare7Kpp/Flare-R/Compound_Flare/000006.png

Archive: Flickr24K.zip
Exists: True
First 10 entries:
Flickr24K/
Flickr24K/100.jpg
Flickr24K/1000.jpg
Flickr24K/10001.jpg
Flickr24K/10002.jpg
Flickr24K/10005.jpg
Flickr24K/10007.jpg
Flickr24K/10008.jpg
Flickr24K/10009.jpg
Flickr24K/10010.jpg


In [ ]:
import os

print("========== FLARE7K++ FOLDERS ==========")

for root, dirs, files in os.walk("/content/flare_data/Flare7Kpp"):
    level = root.replace("/content/flare_data/Flare7Kpp", "").count(os.sep)

    if level <= 3:
        print("  " * level + os.path.basename(root) + "/")

print("\n========== FLICKR24K FOLDERS ==========")

for root, dirs, files in os.walk("/content/flare_data/Flickr24K"):
    level = root.replace("/content/flare_data/Flickr24K", "").count(os.sep)

    if level <= 1:
        print("  " * level + os.path.basename(root) + "/")

========== FLARE7K++ FOLDERS ==========
Flare7Kpp/
  test_data/
    synthetic/
      mask/
      input/
      gt/
    real/
      mask/
      input/
      gt/
  Flare7K/
    Reflective_Flare/
    Scattering_Flare/
      Compound_Flare/
      Core/
      Glare_with_shimmer/
      Light_Source/
      Streak/
  Flare-R/
    Compound_Flare/
    Light_Source/

========== FLICKR24K FOLDERS ==========
Flickr24K/


In [ ]:
import os

FLARE_ROOT = "/content/flare_data/Flare7Kpp"
FLICKR_ROOT = "/content/flare_data/Flickr24K"

print("Flare7K++:", os.path.exists(FLARE_ROOT))
print("Flickr24K:", os.path.exists(FLICKR_ROOT))

print("\nScattering folder:",
      os.path.exists(os.path.join(
          FLARE_ROOT, "Flare7K", "Scattering_Flare",
          "Compound_Flare"
      )))

print("Reflective folder:",
      os.path.exists(os.path.join(
          FLARE_ROOT, "Flare7K", "Reflective_Flare"
      )))

print("Flickr images:", len([
    f for f in os.listdir(FLICKR_ROOT)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]))

Flare7K++: True
Flickr24K: True

Scattering folder: True
Reflective folder: True
Flickr images: 23949


In [ ]:
import sys
import os
import torch
from torchvision import transforms
from torch.utils.data import DataLoader, random_split, Subset

sys.path.append("/content/Flare7K")

from data_loader import Flare_Image_Loader

# Base image transformations
transform_base_256 = transforms.Compose([
    transforms.RandomCrop(
        (256, 256),
        pad_if_needed=True,
        padding_mode="reflect"
    ),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip()
])

# Flare transformations
transform_flare_256 = transforms.Compose([
    transforms.RandomAffine(
        degrees=(0, 360),
        scale=(0.8, 1.5),
        translate=(300/1440, 300/1440),
        shear=(-20, 20)
    ),
    transforms.CenterCrop((256, 256)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip()
])

# Initialize dataset loader
flare_image_loader_256 = Flare_Image_Loader(
    FLICKR_ROOT,
    transform_base_256,
    transform_flare_256,
    mask_type="luminance"
)

# Load scattering flare images
flare_image_loader_256.load_scattering_flare(
    "Flare7K",
    os.path.join(
        FLARE_ROOT,
        "Flare7K",
        "Scattering_Flare",
        "Compound_Flare"
    )
)

# Load reflective flare images
flare_image_loader_256.load_reflective_flare(
    "Flare7K",
    os.path.join(
        FLARE_ROOT,
        "Flare7K",
        "Reflective_Flare"
    )
)

print("\nDataset restoration successful!")
print("Total images:", len(flare_image_loader_256))

sample = flare_image_loader_256[0]

print("Base:", sample[0].shape)
print("Flare:", sample[1].shape)
print("Merged:", sample[2].shape)
print("Mask:", sample[3].shape)

Base Image Loaded with examples: 23949
Scattering Flare Image: Flare7K  is loaded successfully with examples 5000
Now we have 5000 scattering flare images
Reflective Flare Image: Flare7K  is loaded successfully with examples 2000
Now we have 2000 refelctive flare images

Dataset restoration successful!
Total images: 23949
Base: torch.Size([3, 256, 256])
Flare: torch.Size([3, 256, 256])
Merged: torch.Size([3, 256, 256])
Mask: torch.Size([3, 256, 256])


In [ ]:
from torch.utils.data import random_split, DataLoader, Subset

# Recreate original 80:20 split
total_size = len(flare_image_loader_256)

train_size = int(0.8 * total_size)
val_size = total_size - train_size

generator = torch.Generator().manual_seed(42)

train_dataset_256, val_dataset_256 = random_split(
    flare_image_loader_256,
    [train_size, val_size],
    generator=generator
)

# Use the same sample counts as previous training
train_dataset_256 = Subset(train_dataset_256, range(10000))
val_dataset_256 = Subset(val_dataset_256, range(2500))

# DataLoaders
train_loader_256 = DataLoader(
    train_dataset_256,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader_256 = DataLoader(
    val_dataset_256,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Training images:", len(train_dataset_256))
print("Validation images:", len(val_dataset_256))
print("Training batches:", len(train_loader_256))
print("Validation batches:", len(val_loader_256))
print("Batch size:", 8)

Training images: 10000
Validation images: 2500
Training batches: 1250
Validation batches: 313
Batch size: 8


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Double Convolution Block
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


# U-Net Architecture
class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=3):
        super().__init__()

        self.enc1 = DoubleConv(in_channels, 32)
        self.enc2 = DoubleConv(32, 64)
        self.enc3 = DoubleConv(64, 128)
        self.enc4 = DoubleConv(128, 256)

        self.pool = nn.MaxPool2d(kernel_size=2)

        self.bottleneck = DoubleConv(256, 512)

        self.up4 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(512, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(64, 32)

        self.final = nn.Conv2d(32, out_channels, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.bottleneck(self.pool(e4))

        d4 = self.up4(b)
        d4 = self.dec4(torch.cat([d4, e4], dim=1))

        d3 = self.up3(d4)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))

        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return torch.sigmoid(self.final(d1))


# Initialize model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet().to(device)

criterion = nn.L1Loss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

scaler = torch.amp.GradScaler("cuda")

print("U-Net architecture recreated!")
print("Device:", device)
print("Parameters:", sum(p.numel() for p in model.parameters()))

U-Net architecture recreated!
Device: cuda
Parameters: 7766051


In [ ]:
import os

checkpoint_path = (
    "/content/drive/MyDrive/Flare_Removal_Project/"
    "checkpoints/baseline_256/baseline_unet_256_latest.pth"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

# Restore training progress
model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
scaler.load_state_dict(checkpoint["scaler_state_dict"])

saved_epoch = checkpoint["epoch"]
saved_batch = checkpoint["batch_number"]

print("Checkpoint restored successfully!")
print("Completed epoch:", saved_epoch + 1)
print("Last saved batch:", saved_batch)
print("Last saved loss:", checkpoint["loss"])
print("Image size:", checkpoint["image_size"])
print("Batch size:", checkpoint["batch_size"])

Checkpoint restored successfully!
Completed epoch: 4
Last saved batch: 400
Last saved loss: 0.047095887362957
Image size: 256
Batch size: 8


In [ ]:
import os

checkpoint_dir = (
    "/content/drive/MyDrive/Flare_Removal_Project/"
    "checkpoints/baseline_256"
)

os.makedirs(checkpoint_dir, exist_ok=True)

latest_path = os.path.join(
    checkpoint_dir,
    "baseline_unet_256_latest.pth"
)

backup_path = os.path.join(
    checkpoint_dir,
    "baseline_unet_256_backup.pth"
)


def save_checkpoint(epoch, batch_number, loss):

    checkpoint_data = {
        "epoch": epoch,
        "batch_number": batch_number,

        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),

        "loss": loss,
        "total_epochs": 10,
        "image_size": 256,
        "batch_size": 8,

        "train_samples": 10000,
        "val_samples": 2500
    }

    temp_path = latest_path + ".tmp"

    torch.save(checkpoint_data, temp_path)

    if os.path.exists(latest_path):
        os.replace(latest_path, backup_path)

    os.replace(temp_path, latest_path)


print("Checkpoint saving configured!")
print("Latest:", latest_path)
print("Backup:", backup_path)

Checkpoint saving configured!
Latest: /content/drive/MyDrive/Flare_Removal_Project/checkpoints/baseline_256/baseline_unet_256_latest.pth
Backup: /content/drive/MyDrive/Flare_Removal_Project/checkpoints/baseline_256/baseline_unet_256_backup.pth


In [ ]:
EPOCHS = 10
BATCH_SIZE = 8

# Resume from the saved epoch
start_epoch = saved_epoch

# Repeat the interrupted epoch from its beginning
start_batch = 0

print("Training configuration ready!")
print("Total epochs:", EPOCHS)
print("Starting epoch:", start_epoch + 1)
print("Starting batch:", start_batch + 1)
print("Batches per epoch:", len(train_loader_256))
print("Remaining epochs:", EPOCHS - start_epoch)

Training configuration ready!
Total epochs: 10
Starting epoch: 4
Starting batch: 1
Batches per epoch: 1250
Remaining epochs: 7


In [ ]:
for epoch in range(start_epoch, EPOCHS):

    model.train()
    running_loss = 0.0

    print(f"\n========== EPOCH {epoch + 1}/{EPOCHS} ==========")

    for batch_idx, batch in enumerate(train_loader_256):

        inputs = batch[2].to(device, non_blocking=True)
        targets = batch[0].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            outputs = model(inputs)
            loss = criterion(outputs.float(), targets.float())

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

        current_batch = batch_idx + 1

        if current_batch % 100 == 0:

            save_checkpoint(
                epoch,
                current_batch,
                loss.item()
            )

            print(
                f"Epoch [{epoch + 1}/{EPOCHS}] "
                f"Batch [{current_batch}/{len(train_loader_256)}] "
                f"Loss: {loss.item():.6f} | Checkpoint saved"
            )

    train_loss = running_loss / len(train_loader_256)

    # Validation
    model.eval()
    val_loss = 0.0

    with torch.no_grad():

        for batch in val_loader_256:

            inputs = batch[2].to(device, non_blocking=True)
            targets = batch[0].to(device, non_blocking=True)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                outputs = model(inputs)
                loss = criterion(outputs.float(), targets.float())

            val_loss += loss.item()

    val_loss /= len(val_loader_256)

    # Save completed epoch
    save_checkpoint(epoch, len(train_loader_256), train_loss)

    print("\nEpoch completed!")
    print(f"Training Loss: {train_loss:.6f}")
    print(f"Validation Loss: {val_loss:.6f}")
    print("Checkpoint saved successfully!")


========== EPOCH 4/10 ==========
Epoch [4/10] Batch [100/1250] Loss: 0.085058 | Checkpoint saved
Epoch [4/10] Batch [200/1250] Loss: 0.050683 | Checkpoint saved
Epoch [4/10] Batch [300/1250] Loss: 0.038845 | Checkpoint saved
Epoch [4/10] Batch [400/1250] Loss: 0.074893 | Checkpoint saved
Epoch [4/10] Batch [500/1250] Loss: 0.050122 | Checkpoint saved
Epoch [4/10] Batch [600/1250] Loss: 0.051054 | Checkpoint saved
Epoch [4/10] Batch [700/1250] Loss: 0.071044 | Checkpoint saved
Epoch [4/10] Batch [800/1250] Loss: 0.044307 | Checkpoint saved
Epoch [4/10] Batch [900/1250] Loss: 0.039312 | Checkpoint saved
Epoch [4/10] Batch [1000/1250] Loss: 0.059559 | Checkpoint saved
Epoch [4/10] Batch [1100/1250] Loss: 0.046723 | Checkpoint saved
Epoch [4/10] Batch [1200/1250] Loss: 0.050290 | Checkpoint saved

Epoch completed!
Training Loss: 0.050235
Validation Loss: 0.043118
Checkpoint saved successfully!

========== EPOCH 5/10 ==========
Epoch [5/10] Batch [100/1250] Loss: 0.047751 | Checkpoint save